# PEFT: LoRA & QLoRA

**Companion lesson:** https://ml-viz.vercel.app/courses/fine-tuning-alignment/02-peft-lora-qlora

A from-scratch NumPy demo of LoRA — a low-rank update on top of a frozen weight matrix. We'll build a `LoRALinear` layer, train it on a tiny synthetic task, watch parameter counts plummet, and reason about what 4-bit QLoRA quantization buys you.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(0)

## A frozen "pretrained" weight matrix

We're not actually pretraining a real LLM — we just want a fixed `W0` that plays the role of a frozen pretrained projection. The fine-tuning task: nudge the output so it matches a *new* target mapping.

In [ ]:
d = 64                 # input/output dim
n = 256                # samples in the tiny dataset

# "Pretrained" base weights — frozen for the rest of the notebook.
W0 = rng.normal(0, 0.5, size=(d, d)).astype(np.float32)

# The target mapping the fine-tune wants to reach: base + a low-rank update.
# We construct it to *truly* have low intrinsic rank so LoRA can recover it.
r_true = 4
B_true = rng.normal(0, 0.3, size=(d, r_true)).astype(np.float32)
A_true = rng.normal(0, 0.3, size=(r_true, d)).astype(np.float32)
W_target = W0 + B_true @ A_true

# Synthetic dataset: random inputs, targets produced by the *target* mapping.
X = rng.normal(0, 1, size=(n, d)).astype(np.float32)
Y = X @ W_target.T   # row-vector convention: y = x @ W.T

print('frozen W0 shape:', W0.shape, '  true ΔW rank:', r_true)

## The LoRA layer

`LoRALinear` keeps `W0` frozen and exposes two trainable matrices `A` (r×d) and `B` (d×r). The forward pass adds a low-rank correction:

$$
y = x W_0^\top + \frac{\alpha}{r}\, x (B A)^\top.
$$

Standard initialisation: `A` ~ Kaiming-normal (small), `B` = 0. This means the adapter starts as a *no-op*, so the first forward pass exactly matches the base model.

In [ ]:
class LoRALinear:
    def __init__(self, W0, r=4, alpha=8):
        self.W0 = W0                    # frozen base (d_out, d_in)
        d_out, d_in = W0.shape
        self.r = r
        self.alpha = alpha
        self.scale = alpha / r
        # A: small random; B: zeros (adapter starts as identity update of zero).
        self.A = rng.normal(0, 1/np.sqrt(d_in), size=(r, d_in)).astype(np.float32)
        self.B = np.zeros((d_out, r), dtype=np.float32)

    def forward(self, x):
        # y = x W0^T + scale * x (B A)^T
        # cache for backward
        self.x = x
        self.Ax = x @ self.A.T                              # (n, r)
        self.lora_out = self.scale * (self.Ax @ self.B.T)   # (n, d_out)
        self.base_out = x @ self.W0.T
        return self.base_out + self.lora_out

    def backward(self, dY):
        # dL/dB: scale * (Ax)^T @ dY  →  shape (d_out, r) ... but careful with axes:
        # lora_out = scale * Ax @ B^T, dL/dB^T = scale * Ax^T @ dY ⇒ dL/dB = scale * dY^T @ Ax
        dB = self.scale * (dY.T @ self.Ax)                  # (d_out, r)
        # dL/dA: scale * (dY @ B)^T @ x  ⇒ dL/dA = scale * (dY @ B).T @ x
        dA = self.scale * (dY @ self.B).T @ self.x          # (r, d_in)
        return dA, dB

    def trainable_params(self):
        return self.A.size + self.B.size

    def full_ft_params(self):
        return self.W0.size

layer = LoRALinear(W0, r=8, alpha=16)
print(f'trainable LoRA params: {layer.trainable_params()}')
print(f'equivalent full-FT params: {layer.full_ft_params()}')
print(f'savings: {1 - layer.trainable_params()/layer.full_ft_params():.1%}')

## Training the adapter

Mean-squared error on the synthetic targets, vanilla gradient descent on `A` and `B` only. `W0` never moves.

In [ ]:
def train(layer, X, Y, steps=400, lr=0.02):
    losses = []
    for _ in range(steps):
        pred = layer.forward(X)
        diff = pred - Y
        loss = (diff ** 2).mean()
        dY = 2 * diff / diff.size * Y.shape[0]    # mean over batch
        dA, dB = layer.backward(dY)
        layer.A -= lr * dA
        layer.B -= lr * dB
        losses.append(loss)
    return losses

layer = LoRALinear(W0, r=8, alpha=16)
losses = train(layer, X, Y, steps=400, lr=0.02)

plt.plot(losses, color='#6366f1')
plt.xlabel('step'); plt.ylabel('MSE loss')
plt.title('LoRA training — rank 8 on a rank-4 target')
plt.yscale('log')
plt.show()
print(f'final loss: {losses[-1]:.6f}')

## Rank sweep

The target's intrinsic rank is **4**. What happens to final loss as you vary the adapter rank `r`?

In [ ]:
final_losses = {}
for r in [1, 2, 4, 8, 16]:
    layer = LoRALinear(W0, r=r, alpha=16)
    L = train(layer, X, Y, steps=400, lr=0.02)
    final_losses[r] = L[-1]

ranks = list(final_losses.keys())
vals = [final_losses[r] for r in ranks]
plt.bar([str(r) for r in ranks], vals, color='#14b8a6')
plt.xlabel('LoRA rank r'); plt.ylabel('final MSE')
plt.title('Rank below intrinsic rank under-fits; rank ≥ 4 fits well')
plt.yscale('log')
plt.show()
for r, v in final_losses.items():
    print(f'  r = {r:>2d}  →  final MSE = {v:.5f}')

Notice the elbow at `r = 4`. Below the intrinsic rank, the adapter can't represent the true update; above it, you spend extra parameters with diminishing returns. **The rank knob is the capacity dial.**

## A pinch of QLoRA: 4-bit quantization of the frozen base

Real QLoRA uses NF4, a quantile-based 4-bit code tuned for normal-distributed weights. To get the *flavour* without the engineering, we'll do plain symmetric 4-bit linear quantization on `W0` and measure the loss in approximation. The adapters stay in float32 — that's the whole point.

In [ ]:
def quantize_int4(W):
    """Symmetric per-tensor int4 quantization. 4 bits ⇒ 16 levels ⇒ -8..7."""
    scale = np.abs(W).max() / 7.0
    q = np.clip(np.round(W / scale), -8, 7).astype(np.int8)
    return q, scale

def dequantize_int4(q, scale):
    return q.astype(np.float32) * scale

q, s = quantize_int4(W0)
W0_q = dequantize_int4(q, s)
err = np.linalg.norm(W0 - W0_q) / np.linalg.norm(W0)
print(f'frozen-base quantization error: {err:.2%}')
print(f'storage: {W0.nbytes} bytes (fp32) → {q.nbytes//2} bytes (4-bit packed)')

Now retrain the LoRA adapter on top of the *quantized* base. The trainable parameters (A, B) are still float32 — only the storage of the frozen part shrank.

In [ ]:
layer_q = LoRALinear(W0_q, r=8, alpha=16)
losses_q = train(layer_q, X, Y, steps=400, lr=0.02)

plt.plot(losses, label='LoRA on fp32 base', color='#6366f1')
plt.plot(losses_q, label='QLoRA-style on 4-bit base', color='#f97316', linestyle='--')
plt.xlabel('step'); plt.ylabel('MSE loss')
plt.yscale('log'); plt.legend()
plt.title('Quantizing the frozen base barely moves final loss')
plt.show()
print(f'fp32-base final  : {losses[-1]:.5f}')
print(f'int4-base final  : {losses_q[-1]:.5f}')

## ✏️ Your turn

Write `params_saved_pct(d, r)` that returns the percentage of trainable parameters saved when you swap a full $d \times d$ update for a rank-$r$ LoRA adapter. Then verify that for $d = 4096$ and $r = 8$, savings are above 99 %.

In [ ]:
def params_saved_pct(d, r):
    # TODO(you): return 100 * (1 - lora_params / full_params)
    # where full_params = d*d and lora_params = 2*d*r.
    return 0.0

assert abs(params_saved_pct(64, 8) - (1 - 2*64*8/64**2) * 100) < 1e-6
assert params_saved_pct(4096, 8) > 99.0
assert params_saved_pct(1024, 16) > 96.0
print('passed ✓')

<details><summary>Solution</summary>

```python
def params_saved_pct(d, r):
    full = d * d
    lora = 2 * d * r
    return 100.0 * (1.0 - lora / full)
```

</details>